<a href="https://colab.research.google.com/github/exyde01-sketch/Artefactos/blob/main/Menu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install mne

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 46.7 MB/s eta 0:00:00
  Attempting uninstall: decorator
    Found existing installation: decorator 4.4.2
    Uninstalling decorator-4.4.2:
      Successfully uninstalled decorator-4.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [ ]:
import argparse
from pathlib import Path

import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mne

from mne.preprocessing import ICA
from google.colab import files

# ============================================================
# SECCION 1: Cargar CSV e identificar artefactos con
# Extended Infomax ICA
# ============================================================

def cargar_csv_eeg(ruta_csv):
    """Carga un archivo CSV con señales EEG o solicita subirlo si no existe."""
    ruta = Path(ruta_csv)

    if not ruta.exists():
        print(f"El archivo '{ruta_csv}' no fue encontrado en el entorno.")
        print("Por favor, selecciona y sube tu archivo CSV a continuación:")
        subidos = files.upload()
        if not subidos:
            raise FileNotFoundError("No se subió ningún archivo.")
        # Obtener el nombre del archivo recién subido
        nombre_subido = list(subidos.keys())[0]
        ruta = Path(nombre_subido)

    df = pd.read_csv(ruta)

    if df.empty:
        raise ValueError('El archivo CSV está vacío.')

    print(f'Archivo cargado: {ruta}')
    print(f'Forma del dataset: {df.shape}')
    print(f'Columnas: {list(df.columns[:10])}')

    return df, ruta


def identificar_artefactos_ica(
    df_eeg,
    umbral_kurtosis=5.0,
    sfreq=256.0
):
    """
    Identifica posibles artefactos EEG utilizando
    Extended Infomax ICA y curtosis por componente.
    """
    print(
        '--- SECCION 1: IDENTIFICACION DEL ARTEFACTO ---'
    )
    print('Algoritmo: Extended Infomax ICA')

    tiene_timestamp = 'Timestamp' in df_eeg.columns

    if tiene_timestamp:
        datos_sensores = df_eeg.drop(
            columns=['Timestamp']
        )
    else:
        datos_sensores = df_eeg.copy()

    datos_sensores = datos_sensores.select_dtypes(
        include=[np.number]
    )

    if datos_sensores.empty:
        raise ValueError(
            'No se encontraron canales EEG numéricos.'
        )

    X = datos_sensores.to_numpy(dtype=float)

    if not np.isfinite(X).all():
        raise ValueError(
            'Los canales EEG contienen NaN o valores '
            'infinitos. Debes corregirlos antes del ICA.'
        )

    n_muestras, n_canales = X.shape

    if n_muestras < 2:
        raise ValueError(
            'Se necesitan al menos dos muestras.'
        )

    if n_canales < 2:
        raise ValueError(
            'ICA necesita al menos dos canales '
            'para separar componentes.'
        )

    if sfreq <= 0:
        raise ValueError(
            'La frecuencia de muestreo debe ser positiva.'
        )

    X_mean = np.mean(X, axis=0)
    X_std = np.std(X, axis=0)

    canales_constantes = np.where(X_std == 0)[0]

    if len(canales_constantes) > 0:
        nombres = [
            datos_sensores.columns[i]
            for i in canales_constantes
        ]
        raise ValueError(
            'Se encontraron canales constantes: '
            f'{nombres}. Debes eliminarlos antes del ICA.'
        )

    X_norm = (X - X_mean) / X_std

    nombres_canales = [
        str(col) for col in datos_sensores.columns
    ]

    info_mne = mne.create_info(
        ch_names=nombres_canales,
        sfreq=sfreq,
        ch_types=['eeg'] * n_canales
    )

    raw = mne.io.RawArray(
        X_norm.T,
        info_mne,
        verbose=False
    )

    ica = ICA(
        n_components=n_canales,
        method='infomax',
        fit_params={'extended': True},
        random_state=42,
        max_iter=1000,
        verbose=False
    )

    print('Ejecutando Extended Infomax ICA...')

    ica.fit(
        raw,
        picks='eeg',
        verbose=False
    )

    fuentes_independientes = (
        ica.get_sources(raw).get_data().T
    )

    print('ICA finalizado.')
    print(
        'Componentes independientes obtenidos:',
        fuentes_independientes.shape[1]
    )

    componentes_artefactos = []
    kurtosis_por_componente = {}

    for i in range(fuentes_independientes.shape[1]):
        componente = fuentes_independientes[:, i]
        media = np.mean(componente)
        std = np.std(componente)

        if std == 0:
            kurtosis = 0.0
        else:
            kurtosis = (
                np.mean(
                    ((componente - media) / std) ** 4
                ) - 3.0
            )

        kurtosis_por_componente[i] = kurtosis

        if abs(kurtosis) > umbral_kurtosis:
            componentes_artefactos.append(i)
            print(
                f'[!] IC {i}: POSIBLE ARTEFACTO. '
                f'Curtosis: {kurtosis:.2f}'
            )
        else:
            print(
                f'[OK] IC {i}: No supera el umbral. '
                f'Curtosis: {kurtosis:.2f}'
            )

    print(
        'Posibles componentes artefactados:',
        len(componentes_artefactos)
    )

    return {
        'X': X,
        'X_mean': X_mean,
        'X_std': X_std,
        'X_norm': X_norm,
        'nombres_canales': nombres_canales,
        'fuentes_independientes': fuentes_independientes,
        'kurtosis_por_componente': kurtosis_por_componente,
        'componentes_artefactos': componentes_artefactos,
        'ica': ica,
        'raw': raw
    }

# ============================================================
# SECCION 2: Eliminar artefactos con Extended Infomax ICA
# ============================================================

def eliminar_artefactos_ica(
    df_eeg,
    umbral_kurtosis=5.0,
    sfreq=256.0
):
    """
    Excluye los componentes detectados como posibles
    artefactos y reconstruye las señales EEG mediante MNE.
    """
    print(
        '--- SECCION 2: ELIMINACION DEL ARTEFACTO ---'
    )

    info = identificar_artefactos_ica(
        df_eeg,
        umbral_kurtosis=umbral_kurtosis,
        sfreq=sfreq
    )

    ica = info['ica']
    raw = info['raw']
    componentes_artefactos = info['componentes_artefactos']

    ica.exclude = componentes_artefactos.copy()
    print('Componentes excluidos:', ica.exclude)

    raw_limpio = raw.copy()
    ica.apply(
        raw_limpio,
        exclude=componentes_artefactos,
        verbose=False
    )

    X_reconstruido_norm = raw_limpio.get_data().T
    X_reconstruido = (X_reconstruido_norm * info['X_std']) + info['X_mean']

    df_limpio = pd.DataFrame(
        X_reconstruido,
        columns=info['nombres_canales']
    )

    if 'Timestamp' in df_eeg.columns:
        df_limpio.insert(
            0,
            'Timestamp',
            df_eeg['Timestamp'].values
        )

    return df_limpio, componentes_artefactos, info

# ============================================================
# SECCION 3: Graficas de deteccion de artefactos
# ============================================================

def graficar_deteccion_artefactos(
    df_eeg,
    info,
    mostrar=False,
    guardar_png=None
):
    print('--- SECCION 3: GRAFICAS DE DETECCION ---')
    columnas = [col for col in info['nombres_canales']][:6]

    fig, axes = plt.subplots(
        len(columnas) + 1,
        1,
        figsize=(12, 2.8 * (len(columnas) + 1)),
        squeeze=False
    )
    axes = axes.flatten()
    x = np.arange(len(df_eeg))

    for ax, col in zip(axes[:-1], columnas):
        ax.plot(x, df_eeg[col].to_numpy(), linewidth=1.0)
        ax.set_ylabel(col)
        ax.grid(alpha=0.2)

    labels = list(info['kurtosis_por_componente'].keys())
    valores = [info['kurtosis_por_componente'][k] for k in labels]
    artefactos = set(info['componentes_artefactos'])
    colores = ['red' if i in artefactos else 'steelblue' for i in labels]

    ax = axes[-1]
    ax.bar(labels, valores, color=colores)
    umbral = info.get('umbral_kurtosis', 5.0)
    ax.axhline(0, color='black', linewidth=1)
    ax.axhline(umbral, color='orange', linestyle='--', linewidth=1.2, label=f'Umbral = {umbral}')
    ax.axhline(-umbral, color='orange', linestyle='--')
    ax.set_title('Curtosis por componente Extended Infomax ICA')
    ax.set_xlabel('Componente ICA')
    ax.set_ylabel('Curtosis')
    ax.grid(alpha=0.2)
    ax.legend()

    fig.suptitle('Detección de artefactos mediante Extended Infomax ICA')
    fig.tight_layout()

    if guardar_png:
        fig.savefig(guardar_png, dpi=150, bbox_inches='tight')
        print(f'Gráfico guardado en: {guardar_png}')

    if mostrar:
        plt.show()

    plt.close(fig)
    return fig

# ============================================================
# SECCION 4: Comparativa antes y despues
# ============================================================

def graficar_comparativa_antes_despues(
    df_original,
    df_limpio,
    mostrar=False,
    guardar_png=None
):
    print('--- SECCION 4: COMPARATIVA ANTES Y DESPUES ---')
    columnas = [
        col for col in df_original.columns
        if col != 'Timestamp'
        and col in df_limpio.columns
        and pd.api.types.is_numeric_dtype(df_original[col])
    ][:4]

    if not columnas:
        raise ValueError('No hay canales numéricos para comparar.')

    fig, axes = plt.subplots(
        len(columnas),
        2,
        figsize=(14, 3.2 * len(columnas)),
        squeeze=False
    )
    x = np.arange(len(df_original))

    for i, col in enumerate(columnas):
        axes[i, 0].plot(x, df_original[col].to_numpy(), color='tab:blue', linewidth=1.0)
        axes[i, 0].set_title(f'{col} - Antes')
        axes[i, 0].grid(alpha=0.2)

        axes[i, 1].plot(x, df_limpio[col].to_numpy(), color='tab:green', linewidth=1.0)
        axes[i, 1].set_title(f'{col} - Después')
        axes[i, 1].grid(alpha=0.2)

        if i == len(columnas) - 1:
            axes[i, 0].set_xlabel('Muestra')
            axes[i, 1].set_xlabel('Muestra')

    fig.suptitle('Comparativa EEG antes y después de Extended Infomax ICA')
    fig.tight_layout()

    if guardar_png:
        fig.savefig(guardar_png, dpi=150, bbox_inches='tight')
        print(f'Gráfico comparativo guardado en: {guardar_png}')

    if mostrar:
        plt.show()

    plt.close(fig)
    return fig

# ============================================================
# SECCION 5: Procesamiento completo del archivo
# ============================================================

def procesar_archivo_csv(
    ruta_csv,
    umbral_kurtosis=5.0,
    mostrar_graficos=False,
    sfreq=256.0
):
    """
    Procesa el CSV:
    1. Carga las señales (o las solicita subir si no existen).
    2. Ejecuta Extended Infomax ICA.
    3. Identifica posibles artefactos.
    4. Reconstruye las señales.
    5. Genera las gráficas.
    6. Exporta el CSV limpio.
    """
    df_original, ruta_final = cargar_csv_eeg(ruta_csv)

    (
        df_limpio,
        artefactos,
        info
    ) = eliminar_artefactos_ica(
        df_original,
        umbral_kurtosis=umbral_kurtosis,
        sfreq=sfreq
    )

    info['umbral_kurtosis'] = umbral_kurtosis

    carpeta_salida = ruta_final.parent
    nombre_base = ruta_final.stem

    graficar_deteccion_artefactos(
        df_original,
        info,
        mostrar=mostrar_graficos,
        guardar_png=(
            carpeta_salida
            / f'{nombre_base}_deteccion_artefactos.png'
        )
    )

    graficar_comparativa_antes_despues(
        df_original,
        df_limpio,
        mostrar=mostrar_graficos,
        guardar_png=(
            carpeta_salida
            / f'{nombre_base}_comparativa_antes_despues.png'
        )
    )

    salida_csv = (
        carpeta_salida
        / f'{nombre_base}_limpio.csv'
    )

    df_limpio.to_csv(
        salida_csv,
        index=False
    )

    print(f'CSV limpio guardado en: {salida_csv}')
    print('Componentes excluidos:', artefactos)

    return df_limpio, artefactos, salida_csv

# ============================================================
# SECCION 6: Ejecucion desde la terminal
# ============================================================

if __name__ == '__main__':
    parser = argparse.ArgumentParser(
        description='Procesamiento EEG mediante Extended Infomax ICA.'
    )
    parser.add_argument(
        'csv',
        help='Ruta del archivo CSV con señales EEG.'
    )
    parser.add_argument(
        '--umbral',
        type=float,
        default=5.0,
        help='Umbral absoluto de curtosis excedente. Valor predeterminado: 5.0.'
    )
    parser.add_argument(
        '--sfreq',
        type=float,
        default=256.0,
        help='Frecuencia de muestreo EEG en Hz. Valor predeterminado: 256.'
    )
    parser.add_argument(
        '--mostrar',
        action='store_true',
        help='Muestra las gráficas al finalizar.'
    )

    # En entornos interactivos (como notebooks), evitamos fallar al parsear argumentos de sys.argv
    import sys
    if 'ipykernel' not in sys.modules:
        args = parser.parse_args()
        procesar_archivo_csv(
            args.csv,
            umbral_kurtosis=args.umbral,
            mostrar_graficos=args.mostrar,
            sfreq=args.sfreq
        )

ModuleNotFoundError: No module named 'mne'

In [1]:

def mostrar_menu():
    print("\n" + "=" * 50)
    print("       MENÚ DE PROCESAMIENTO DE SEÑALES")
    print("=" * 50)
    print("1. Equipo 1 - Fast_ICA")
    print("2. Equipo 2 - Wavelet_ICA")
    print("3. Equipo 3 - Pasaband")
    print("4. Equipo 4 - InterferenciaModifier")
    print("5. Equipo 5 - ICA")
    print("6. Equipo 6 - ExtendInfomax_ICA")
    print("7. Equipo 7 - Artefacto_NoRaw")
    print("0. Salir")
    print("=" * 50)


def ejecutar_equipo(opcion):
    temas = {
        "1": ("Equipo 1", "Fast_ICA"),
        "2": ("Equipo 2", "Wavelet_ICA"),
        "3": ("Equipo 3", "Pasaband"),
        "4": ("Equipo 4", "InterferenciaModifier"),
        "5": ("Equipo 5", "ICA"),
        "6": ("Equipo 6", "ExtendInfomax_ICA"),
        "7": ("Equipo 7", "Artefacto_NoRaw")
    }

    if opcion in temas:
        equipo, tema = temas[opcion]
        print("\n" + "-" * 40)
        print(f"Seleccionaste: {equipo}")
        print(f"Tema: {tema}")
        print("-" * 40)

        # Aquí puedes agregar el código específico
        # para ejecutar el procesamiento de cada tema.
    else:
        print("Opción no válida. Intenta nuevamente.")


def main():
    while True:
        mostrar_menu()
        opcion = input("Selecciona una opción (0-7): ").strip()

        # --- EQUIPO 6 ---
        if opcion == "6":
            print("Asegúrate de haber subido tu archivo CSV al entorno.")
            ruta_csv = input("Ingresa el nombre o ruta de tu archivo CSV (ej. datos.csv): ").strip()

            try:
                # Se llama a la función principal
                procesar_archivo_csv(ruta_csv, umbral_kurtosis=5.0, mostrar_graficos=True)
            except Exception as e:
                print(f"Error al procesar el algoritmo: {e}")
        # ---------------------------------------------

        if opcion == "0":
            print("Saliendo del programa. ¡Hasta pronto!")
            break

        ejecutar_equipo(opcion)


if __name__ == "__main__":
    main()



       MENÚ DE PROCESAMIENTO DE SEÑALES
1. Equipo 1 - Fast_ICA
2. Equipo 2 - Wavelet_ICA
3. Equipo 3 - Pasaband
4. Equipo 4 - InterferenciaModifier
5. Equipo 5 - ICA
6. Equipo 6 - ExtendInfomax_ICA
7. Equipo 7 - Artefacto_NoRaw
0. Salir
Selecciona una opción (0-7): 6
Asegúrate de haber subido tu archivo CSV al entorno.
Ingresa el nombre o ruta de tu archivo CSV (ej. datos.csv): 0
Error al procesar el algoritmo: name 'procesar_archivo_csv' is not defined

----------------------------------------
Seleccionaste: Equipo 6
Tema: ExtendInfomax_ICA
----------------------------------------

       MENÚ DE PROCESAMIENTO DE SEÑALES
1. Equipo 1 - Fast_ICA
2. Equipo 2 - Wavelet_ICA
3. Equipo 3 - Pasaband
4. Equipo 4 - InterferenciaModifier
5. Equipo 5 - ICA
6. Equipo 6 - ExtendInfomax_ICA
7. Equipo 7 - Artefacto_NoRaw
0. Salir
Selecciona una opción (0-7): 0
Saliendo del programa. ¡Hasta pronto!
